# PACE Sarvam Translate Quickstart

This notebook shows how to run `sarvamai/sarvam-translate` with PACE for a simple translation task.

> This model is used through its chat template, with a translation instruction in the system message and the source text in the user message.

## Usage Notes

- `sarvamai/sarvam-translate` expects chat-style prompting.
- The quickstart pattern from the model card uses a system message like `Translate the text below to Hindi.`
- This notebook keeps generation deterministic with `do_sample=False` for a simple translation example.

## Runtime Notes

- Backend/cache in this notebook: `BMC` + `JIT`
- Import `torch` before `pace`.
- Check `../docs/PerformanceGuide.md` for runtime tips.

In [ ]:
import torch
from transformers import AutoTokenizer

import pace  # noqa: F401
from pace.llm import (
    LLMModel,
    SamplingConfig,
    KVCacheType,
    LLMOperatorType,
    OperatorConfig,
)
from pace.llm.attention import AttentionBackendType
from pace.utils.logging import suppress_logging

## Small Note on Prompting

- `build_translation_prompt(...)` applies the tokenizer chat template.
- The target language goes in the system message.
- The source text goes in the user message.

In [ ]:
def build_translation_prompt(tokenizer, source_text, target_language):
    messages = [
        {"role": "system", "content": f"Translate the text below to {target_language}."},
        {"role": "user", "content": source_text},
    ]
    if hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return f"Translate the text below to {target_language}.\n\n{source_text}"

In [ ]:
model_name = "sarvamai/sarvam-translate"
torch_dtype = torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

opconfig = OperatorConfig(**{LLMOperatorType.Attention: AttentionBackendType.JIT})

## Model Load Parameters

- Load the model with `dtype=torch.bfloat16`.
- This notebook uses `KVCacheType.BMC`.
- Attention backend is set through `OperatorConfig` with `AttentionBackendType.JIT`.
- `suppress_logging()` is used during model load to keep the notebook output clean.

In [ ]:
with suppress_logging():
    pace_model = LLMModel(
        model_name,
        dtype=torch_dtype,
        kv_cache_type=KVCacheType.BMC,
        opconfig=opconfig,
    )

## Translation Example

This example follows the model-card quickstart shape, but runs through PACE.

It translates one English sentence into Hindi, Tamil, Malayalam, Telugu, and Kannada.


In [ ]:
target_languages = ["Hindi", "Tamil", "Malayalam", "Telugu", "Kannada"]
source_text = (
    "AMD builds thoughtful, high-performance technology, and it is exciting to see "
    "its tools help developers create faster, smarter AI systems."
)

In [ ]:
translation_prompts = [
    build_translation_prompt(tokenizer, source_text, target_language)
    for target_language in target_languages
]
encoded_inputs = [
    tokenizer([prompt], return_tensors="pt", padding="longest")
    for prompt in translation_prompts
]


In [ ]:
sampling_config = SamplingConfig(
    max_new_tokens=256,
    do_sample=False,
)

In [ ]:
translations = {}
with suppress_logging():
    for target_language, input_encoded in zip(target_languages, encoded_inputs):
        outputs = pace_model.generate(input_encoded, sampling_config)
        generated_ids = outputs.output_token_ids[0, input_encoded.input_ids.shape[1]:].tolist()
        translations[target_language] = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

In [ ]:
print("Input:", source_text)
for target_language in target_languages:
    print(f"\n{target_language}:")
    print(translations[target_language])